# Optimal pointing calculation for NIRCam

In this notebook we will:

1. Download an existing JWST observation to use as a reference DQ map and PSF
2. Find optimal regions for the 10110 SURVEY Cycle 5 program based on the dither pattern
3. Visualize the optimal regions and pick one pointing
4. Check the short wavelength channel
5. Calculate the required offset to enter in APT for that pointing

This is a streamlined version of `understand_pointing.ipynb`, in which I did a lot explanation and exploration.

## Tunable parameters

In [ ]:
FORBIDDEN_SIZE = 14
TARGET_2743 = "WISE-1206"
CROP_SIZE = 70
KERNEL = "weighted"
MIN_EDGE_DISTANCE = 500
NDITHERS = 5

hs = CROP_SIZE // 2

## Downloading an existing observation

We will download an existing observation from the Cycle 1 GO 2473 program.

In [ ]:
from pathlib import Path
from astroquery.mast import Observations
from pandas import read_csv

pos_df = read_csv("../resources/position_file_2473.csv", comment="#")
target = TARGET_2743  # we pick a target with primary dithers
target_df = pos_df.query("target==@target")
target_df_lw = target_df[target_df.file.str.endswith("long")].sort_values("file").reset_index()
target_df_lw

file_row = target_df_lw.iloc[0]
file_id = file_row.file
file_id_nodet = "_".join(file_id.split("_")[:-1])
x, y = file_row.x, file_row.y
uri_lw = f"mast:JWST/product/{file_id}_cal.fits"
data_path = Path("./data")
data_path.mkdir(exist_ok=True)
path_lw = data_path / uri_lw.split("/")[-1]

Observations.download_file(uri_lw, local_path=path_lw);

Let us extract the header and science data, and derive the DQ map.
We can then take a quick look at the data.

In [ ]:
from astropy.io import fits
import numpy as np

with fits.open(path_lw) as hdul:
    hdr = hdul[0].header
    img = hdul["SCI"].data
dq_mask = np.isnan(img)

In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt

img_crop = img[y - hs: y + hs, x - hs: x + hs]

rcParams["image.origin"] = "lower"

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axf, axz = axs

im = axf.imshow(img, norm="symlog")
axf.plot(x, y, "r*", label="Target position")
axf.set_title(f"Reference data of {target} from GO 2473")
axf.set_xlabel("X [pix]")
axf.set_ylabel("Y [pix]")
fig.colorbar(im, label="DN/s")
axf.legend()

im = axz.imshow(img_crop, norm="symlog")
axz.set_title(f"Zoom on {target}")
axz.set_xlabel("X [pix]")
axz.set_ylabel("Y [pix]")
fig.colorbar(im, label="DN/s")

plt.show()

Finally, we will filter the cropped image to remove the bad pixels
and have a reference PSF for display and region search purposes.

In [ ]:
from coldest.pointing import filter_crop
psf = filter_crop(img_crop)

## Searching the optimal region

### Getting dither positions

We use the dither offsets from the [dither file](https://jwst-docs.stsci.edu/files/216457358/216457359/2/1763675557795/NIRCamDitherPatterns.zip)
downloadable from [JDox](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-operations/nircam-dithers-and-mosaics/nircam-primary-dithers#gsc.tab=0).

We already copied the `INTRAMODULEBOX` file to `resources`.

In [ ]:
import pandas as pd

dither_df = pd.read_csv(
    "../resources/NircamImagingIntramoduleBox.txt",
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=["x_docs", "y_docs"],
    index_col=0,
).reset_index(drop=True)[:NDITHERS]
target_df_lw = pd.concat([target_df_lw, dither_df], axis=1)
target_df_lw = target_df_lw.loc[:, ~target_df_lw.columns.duplicated()]

In [ ]:
from coldest.pointing import PSCALE_DICT
pscale = PSCALE_DICT[hdr["DETECTOR"]]

x_dithers = (target_df_lw.x_docs / pscale).values
y_dithers = (target_df_lw.y_docs / pscale).values
dither_offsets = [xy for xy in zip(x_dithers, y_dithers)]

### Running the search

In [ ]:
from coldest.pointing import do_region_search

best_x_dith, best_y_dith = do_region_search(
    dq_mask,
    img,
    CROP_SIZE,
    psf,
    show=True,
    joint_offsets=dither_offsets,
    forbidden_size=FORBIDDEN_SIZE,
    kernel=KERNEL,
    min_edge_distance=MIN_EDGE_DISTANCE,
)

### Visualizing best pointing

In [ ]:
from coldest.pointing import plot_dithers

for i in range(NDITHERS):
    xopt, yopt = best_x_dith[i], best_y_dith[i]
    xopt_all, yopt_all = xopt + x_dithers.astype(int), yopt + y_dithers.astype(int)
    fig_full, fig_zoom = plot_dithers(img, xopt_all, yopt_all, CROP_SIZE, psf)
    fig_full.axes[0].set_title(f"Pointing {i+1}: Dithers on the full frame")
    fig_zoom.suptitle(f"Pointing {i+1}: Zoom on each dither")
    plt.show()

## Selecting a pointing and calculating the offset

In [ ]:
opt_idx = 1

xopt, yopt = best_x_dith[opt_idx], best_y_dith[opt_idx]
xopt_all, yopt_all = xopt + x_dithers.astype(int), yopt + y_dithers.astype(int)

In [ ]:
from coldest.pointing import calculate_offset

xoff, yoff = calculate_offset(xopt, yopt, path_lw)

print(f"Desired offset in arcsec: {xoff:.3f}, {yoff:.3f}")

## Checking the short wavelength channel

In [ ]:
target_df_sw = target_df[target_df.file.str.endswith("nrcb2")].sort_values("file").reset_index()
file_id = target_df_sw.iloc[0].file
file_id_sw = file_id
file_ids = [file_id.replace("nrcb2", f"nrcb{i}") for i in range(1, 5)]
path_sw_list = []
for file_id in file_ids:
    uri_sw = f"mast:JWST/product/{file_id}_cal.fits"
    path_sw = data_path / uri_sw.split("/")[-1]
    path_sw_list.append(path_sw)
    Observations.download_file(uri_sw, local_path=path_sw);

In [ ]:
path_sw = data_path / (file_id_sw + "_cal.fits")
img_sw = fits.open(path_sw)["SCI"].data
row_sw = target_df_sw.iloc[0]
x_sw, y_sw = row_sw.x, row_sw.y

img_crop_sw = img_sw[y_sw - hs: y_sw + hs, x_sw - hs: x_sw + hs]
psf_sw = filter_crop(img_crop_sw)

In [ ]:
npix = img.shape[0]
top = yopt > npix // 2
right = xopt > npix // 2
if top and right:
    det_sw = "nrcb1"
elif top and not right:
    det_sw = "nrcb3"
elif not top and right:
    det_sw = "nrcb2"
elif not top and not right:
    det_sw = "nrcb4"

path_sw = data_path / f"{file_id_nodet}_{det_sw}_cal.fits"

In [ ]:
from coldest.pointing import long_to_short

xopt_all_sw, yopt_all_sw = zip(*[long_to_short(xi, yi, path_lw, path_sw) for xi, yi in zip(xopt_all, yopt_all)])

In [ ]:
fig_full, fig_zoom = plot_dithers(img_sw, xopt_all_sw, yopt_all_sw, CROP_SIZE, psf=psf_sw)
fig_full.axes[0].set_title("Dithers on the full frame (Short-Wavelength)")
fig_zoom.suptitle("Zoom on each dither (Short-Wavelength)")
plt.show()